# Exit-agreement hybrid grid search with a 16K hard stop

This standalone notebook reuses the record normalization and exit-agreement simulator from `EDA_hung.ipynb`. Edit the paths, model filter, method sets, and agreement fractions below. Each policy votes through 8K and returns the recorded `baseline_16K` answer if no quorum forms. The output compares question-balanced accuracy and token cost, then averages benchmarks equally within each model. No Pareto front or detailed benchmark accuracy-versus-token plots are generated.


In [1]:
from pathlib import Path
from itertools import product
import json
import re
import numpy as np
import pandas as pd

FAILED_THINKING_TOKENS = 32_000
INPUT_DIR = Path("examples/fix")
OUTPUT_DIR = Path("output/hybrid_grid_search")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Empty means all models found in INPUT_DIR.
MODEL_CHOICES = []  # e.g. ["gpt_oss_20b_high", "qwen35_4b"]
AGREEMENT_CUTOFF = 8_192
HARD_STOP_METHOD = "baseline_16K"
STAGES = (1_024, 2_048, 4_096)
AGREEMENT_FRACTIONS = (0.5, 0.75, 1.0)
METHOD_SETS = {
    "all_four": ("codestop", "dtsr", "deer", "entrocut"),
    "without_codestop": ("dtsr", "deer", "entrocut"),
    "without_dtsr": ("codestop", "deer", "entrocut"),
    "with_terminator": ("codestop", "dtsr", "deer", "entrocut", "terminator_hf"),
    "deer_entrocut": ("deer", "entrocut"),
}


In [2]:
def repair_json_line(line: str) -> str:
    return re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", line)

def load_jsonl(path: Path):
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: row = json.loads(line)
            except json.JSONDecodeError:
                try: row = json.loads(repair_json_line(line))
                except Exception:
                    bad += 1
                    continue
            if not isinstance(row, dict) or row.get("num_token") != 0:
                rows.append(row)
    return rows, bad

def failed_method_result(method, failure_reason, failure_tokens):
    return {
        "method": method, "position": failure_tokens, "stopped_early": False,
        "correct": False, "finish_reason": "failed",
        "thinking_tokens": FAILED_THINKING_TOKENS, "answer_tokens": 0,
        "completion_tokens": failure_tokens, "prompt_tokens": 0,
        "verifier_score": None, "verifier_prompt_tokens": 0,
        "verifier_completion_tokens": 0,
        "logs": {"generation_failed": True, "failure_reason": failure_reason},
    }

def get_method_results(record):
    """Return method results from normalized, current, or legacy runs."""
    if not isinstance(record, dict):
        return []
    additional = record.get("additional_logs")
    if isinstance(additional, dict):
        method_results = additional.get("method_results")
        if isinstance(method_results, list):
            return method_results
    method_results = record.get("method_results")
    return method_results if isinstance(method_results, list) else []

def iter_method_results(records):
    """Yield valid method-result dictionaries with their parent run."""
    for record_index, record in enumerate(records):
        if not isinstance(record, dict):
            continue
        for method_result in get_method_results(record):
            if isinstance(method_result, dict):
                yield record_index, record, method_result

def process_method_result(record, method_result):
    """Normalize fields shared by every method-level analysis."""
    logs = method_result.get("logs")
    logs = logs if isinstance(logs, dict) else {}
    generation_failed = bool(logs.get("generation_failed"))
    failure_tokens = (
        logs.get("failure_max_completion_token")
        or method_result.get("completion_tokens")
        or method_result.get("position")
        or record.get("num_token")
        or 0
    )
    completion_tokens = (
        failure_tokens if generation_failed
        else method_result.get("completion_tokens") or 0
    )
    answer_tokens = (
        0 if generation_failed else method_result.get("answer_tokens") or 0
    )
    verifier_prompt_tokens = (
        0 if generation_failed
        else method_result.get("verifier_prompt_tokens") or 0
    )
    verifier_completion_tokens = (
        0 if generation_failed
        else method_result.get("verifier_completion_tokens") or 0
    )
    thinking_tokens = (
        FAILED_THINKING_TOKENS if generation_failed
        else method_result.get("thinking_tokens") or 0
    )
    return {
        "method": method_result.get("method"),
        "position": (
            failure_tokens if generation_failed else method_result.get("position")
        ),
        "stopped_early": (
            False if generation_failed else bool(method_result.get("stopped_early"))
        ),
        "correct": (
            False if generation_failed else bool(method_result.get("correct"))
        ),
        "thinking_tokens": thinking_tokens,
        "answer_tokens": answer_tokens,
        "completion_tokens": completion_tokens,
        "final_answer_tokens": (
            answer_tokens if answer_tokens > 0 else completion_tokens
        ),
        "prompt_tokens": (
            0 if generation_failed else method_result.get("prompt_tokens") or 0
        ),
        "verifier_prompt_tokens": verifier_prompt_tokens,
        "verifier_completion_tokens": verifier_completion_tokens,
        "verifier_score": method_result.get("verifier_score"),
        "generation_failed": generation_failed,
        "failure_reason": logs.get("failure_reason"),
        "num_probe_scores": len(logs.get("probe_scores", [])),
        "logs": logs,
    }

def normalize_record(r):
    if not isinstance(r, dict): return r
    if "additional_logs" not in r or not isinstance(r["additional_logs"], dict):
        r["additional_logs"] = {}
    additional = r["additional_logs"]
    # Current replay files store method_results at the top level; older
    # files stored them under additional_logs. Normalize both layouts.
    method_results = get_method_results(r)
    method_name = r.get("agent") or r.get("method") or "unknown"
    num_token = r.get("num_token") or 0
    correct_val = r.get("correct"); 
    if correct_val is None: correct_val = r.get("ok")
    if correct_val is None: correct_val = False
    failure_reason = r.get("failure_reason") or additional.get("failure_reason")
    generation_failed = bool(r.get("generation_failed") or additional.get("generation_failed") or failure_reason == "full_generation_reached_token_limit")
    # The top-level record is one run of the agent, even when it also
    # contains replayed method results. Keep it exactly once.
    if not any(m.get("method") == method_name for m in method_results if isinstance(m, dict)):
        if generation_failed:
            failure_tokens = additional.get("failure_max_completion_token") or num_token
            method_results.append(failed_method_result(method_name, failure_reason, failure_tokens))
        else:
            method_results.append({
            "method": method_name,
            "position": r.get("position") or num_token,
            "stopped_early": r.get("stopped_early") or False,
            "correct": bool(correct_val),
            "finish_reason": r.get("finish_reason") or "stop",
            # Preserve missing thinking usage. Agreement simulation
            # falls back to position/completion_tokens for providers
            # that bake reasoning into the completion count.
            "thinking_tokens": r.get("thinking_tokens"),
            "answer_tokens": r.get("answer_tokens") or 0,
            "completion_tokens": r.get("completion_tokens") or num_token,
            "prompt_tokens": r.get("prompt_tokens") or 0,
            "verifier_score": r.get("verifier_score"),
            "verifier_prompt_tokens": r.get("verifier_prompt_tokens") or 0,
            "verifier_completion_tokens": r.get("verifier_completion_tokens") or 0,
            "logs": {
                "probe_scores": additional.get("probe_scores", []),
                "puma_trials": additional.get("puma_trials", []),
                "entrocut_checks": additional.get("entrocut_checks", []),
                "sufficiency_attempts": additional.get("sufficiency_attempts", []),
                "generation_failed": generation_failed,
                "failure_reason": failure_reason,
                "failure_max_completion_token": r.get("failure_max_completion_token") or additional.get("failure_max_completion_token"),
            },
            })
    additional["method_results"] = method_results
    return r

CODE_BENCHMARKS = {"humaneval", "livecodebench"}
EXCLUDED_CODE_LOG_METHODS = {"dynasor", "answer-coverage", "puma"}


def has_excluded_code_log(record, benchmark):
    """Flag code records containing methods not intended for code analysis."""
    if benchmark not in CODE_BENCHMARKS:
        return False
    return any(
        isinstance(method_result, dict)
        and method_result.get("method") in EXCLUDED_CODE_LOG_METHODS
        for method_result in get_method_results(record)
    )


def glob_all_outputs(base_dir="examples/fix"):
    path = Path(base_dir)
    if not path.is_dir():
        print(f"Output directory not found: {path}")
        return []
    jsonl_files = sorted(path.rglob("*.jsonl"))
    print(f"Found {len(jsonl_files)} JSONL files")
    records = []
    parse_failures = 0
    token_limit_records = 0
    token_limit_cases = set()
    excluded_code_records = 0
    for f in jsonl_files:
        benchmark = f.parent.name
        rows, bad = load_jsonl(f)
        parse_failures += bad
        for li, r in enumerate(rows, 1):
            if isinstance(r, dict):
                line_index = r["line_number"] if r.get("line_number") is not None else li
                additional = r.get("additional_logs") if isinstance(r.get("additional_logs"), dict) else {}
                failure_reason = r.get("failure_reason") or additional.get("failure_reason")
                if failure_reason == "full_generation_reached_token_limit":
                    token_limit_records += 1
                    token_limit_cases.add((benchmark, r.get("model"), line_index))
                has_mr = (
                    isinstance(r.get("method_results"), list)
                    or (
                        isinstance(r.get("additional_logs"), dict)
                        and isinstance(r["additional_logs"].get("method_results"), list)
                    )
                )
                is_sa = ("agent" in r and ("correct" in r or "ok" in r))
                if has_mr or is_sa:
                    if has_excluded_code_log(r, benchmark):
                        excluded_code_records += 1
                        continue
                    r["benchmark"] = benchmark
                    r["source_file"] = f.name
                    r["line_index"] = line_index
                    records.append(normalize_record(r))
    # A failed full-generation run has no nested method results. Add a
    # failed result for every method configured for that benchmark/model.
    expected_methods = {}
    for r in records:
        key = (r.get("benchmark"), r.get("model"))
        methods = expected_methods.setdefault(key, set())
        methods.update(
            m.get("method") for m in get_method_results(r)
            if isinstance(m, dict) and m.get("method")
        )
    for r in records:
        additional = r["additional_logs"]
        failure_reason = r.get("failure_reason") or additional.get("failure_reason")
        generation_failed = bool(r.get("generation_failed") or additional.get("generation_failed") or failure_reason == "full_generation_reached_token_limit")
        if not generation_failed:
            continue
        method_results = get_method_results(r)
        present = {m.get("method") for m in method_results}
        failure_tokens = additional.get("failure_max_completion_token") or r.get("num_token") or 0
        key = (r.get("benchmark"), r.get("model"))
        for method in sorted(expected_methods.get(key, set()) - present):
            method_results.append(failed_method_result(method, failure_reason, failure_tokens))
    print(
        "Excluded code records containing dynasor, answer-coverage, "
        f"or puma logs: {excluded_code_records}"
    )
    print(f"Unrecoverable JSON parse failures: {parse_failures}")
    print(f"Token-limit failure records: {token_limit_records}")
    print(f"Unique question/model cases with a token-limit failure: {len(token_limit_cases)}")
    print(f"Loaded {len(records)} evaluation records.")
    return records

In [3]:
ANSWER_FIELDS = (
    "normalized_answer",
    "extracted_answer",
    "final_answer",
    "answer",
    "prediction",
    "pred",
    "response",
    "output",
    "generated_answer",
)


def first_not_none(*values):
    for value in values:
        if value is not None:
            return value

    return None


def extract_answer(method_result, record):
    """
    Extract the actual answer produced by a method.

    Extend ANSWER_FIELDS when the raw files use another field name.
    """

    method_logs = (
        method_result.get("logs")
        if isinstance(method_result.get("logs"), dict)
        else {}
    )

    containers = [
        method_result,
        method_logs,
    ]

    # Only use the top-level answer when this method was the
    # top-level agent that generated the record.
    top_level_method = (
        record.get("agent")
        or record.get("method")
    )

    if method_result.get("method") == top_level_method:
        containers.append(record)

        additional = record.get("additional_logs")

        if isinstance(additional, dict):
            containers.append(additional)

    for container in containers:
        for field in ANSWER_FIELDS:
            value = container.get(field)

            if value is not None:
                return value

    return None


def normalize_answer(answer, benchmark=None):
    """
    Conservative normalization for the selected output record.

    For code benchmarks, preserve code structure.
    For other benchmarks, normalize capitalization and whitespace.

    Replace this function with benchmark-specific canonicalization
    when one is available.
    """

    if answer is None:
        return None

    if isinstance(answer, (dict, list, tuple)):
        return json.dumps(
            answer,
            sort_keys=True,
            ensure_ascii=False,
        )

    text = str(answer).replace("\r\n", "\n").strip()

    if not text:
        return None

    # Remove an outer Markdown code fence only.
    if text.startswith("```") and text.endswith("```"):
        text = re.sub(
            r"^```[a-zA-Z0-9_+-]*\s*",
            "",
            text,
        )
        text = re.sub(
            r"\s*```$",
            "",
            text,
        ).strip()

    benchmark = str(benchmark or "").lower()

    is_code_benchmark = any(
        name in benchmark
        for name in (
            "humaneval",
            "livecodebench",
            "code",
        )
    )

    if is_code_benchmark:
        # Remove trailing spaces without collapsing code lines.
        return "\n".join(
            line.rstrip()
            for line in text.splitlines()
        ).strip()

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip().lower()


def build_exit_dataframe(records):
    """
    Build one row per benchmark/model/question/run/method.

    This differs from process_records() because it retains:
      - run_id
      - actual answer
      - normalized selected-output representation
    """

    rows = []

    for record_index, record in enumerate(records):
        additional = (
            record.get("additional_logs")
            if isinstance(
                record.get("additional_logs"),
                dict,
            )
            else {}
        )

        method_results = get_method_results(record)

        benchmark = record.get("benchmark")
        model = record.get("model")
        line_index = record.get(
            "line_index",
            record_index,
        )
        source_file = record.get(
            "source_file",
            "unknown_file",
        )

        sample_id = (
            f"{benchmark}_{line_index}"
        )

        # Distinguishes repeated runs of the same question.
        run_id = (
            f"{benchmark}|{model}|"
            f"{source_file}|{line_index}|{record_index}"
        )

        for method_result in method_results:
            if not isinstance(method_result, dict):
                continue

            values = process_method_result(record, method_result)
            logs = values["logs"]
            generation_failed = values["generation_failed"]
            thinking_tokens = float(values["thinking_tokens"])
            completion_tokens = float(values["completion_tokens"])
            verifier_prompt_tokens = float(
                values["verifier_prompt_tokens"]
            )
            verifier_completion_tokens = float(
                values["verifier_completion_tokens"]
            )
            correct = values["correct"]
            stopped_early = values["stopped_early"]
            raw_answer_tokens = method_result.get("answer_tokens")
            answer_tokens = float(
                values["answer_tokens"]
                if generation_failed or raw_answer_tokens is not None
                else max(completion_tokens - thinking_tokens, 0.0)
            )
            answer = (
                None
                if generation_failed
                else extract_answer(method_result, record)
            )

            verifier_total_tokens = (
                verifier_prompt_tokens
                + verifier_completion_tokens
            )

            rows.append({
                "benchmark": benchmark,
                "model": model,
                "sample_id": sample_id,
                "run_id": run_id,
                "method": method_result.get(
                    "method"
                ),

                "answer": answer,
                "answer_norm": normalize_answer(
                    answer,
                    benchmark,
                ),

                "correct": correct,
                "stopped_early": stopped_early,
                "generation_failed": (
                    generation_failed
                ),

                "thinking_tokens": (
                    thinking_tokens
                ),
                "completion_tokens": (
                    completion_tokens
                ),
                "answer_tokens": answer_tokens,

                "verifier_prompt_tokens": (
                    verifier_prompt_tokens
                ),
                "verifier_completion_tokens": (
                    verifier_completion_tokens
                ),
                "verifier_logs": logs,
                "verifier_total_tokens": (
                    verifier_total_tokens
                ),

                "total_cost_tokens_est": (
                    completion_tokens
                    + verifier_completion_tokens
                ),
            })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    unit_method = [
        "benchmark",
        "model",
        "sample_id",
        "run_id",
        "method",
    ]

    duplicate_count = int(
        result.duplicated(
            unit_method,
            keep=False,
        ).sum()
    )

    if duplicate_count:
        raise ValueError(
            f"Found {duplicate_count} duplicate "
            "run-method rows."
        )

    return result

In [4]:
def resolve_hard_stop_rows(
    exit_df,
    hard_stop_method=HARD_STOP_METHOD,
):
    """Return at most one exact hard-stop row per repeated run."""
    unit_cols = ["benchmark", "model", "sample_id", "run_id"]
    hard_stop = exit_df[
        exit_df["method"].eq(hard_stop_method)
    ].copy()
    hard_stop["selected_hard_stop_method"] = hard_stop_method

    duplicated = hard_stop.duplicated(unit_cols, keep=False)
    if duplicated.any():
        raise ValueError(
            "Hard-stop resolution produced multiple rows for the same "
            "repeated run."
        )
    return hard_stop

VERIFIER_ATTEMPT_LOG_KEYS = (
    "probe_attempts",
    "sufficiency_attempts",
    "puma_trials",
    "answer_coverage_attempts",
    "dynasor_attempts",
)


def verifier_completion_tokens_through(logs, stop_position, total_tokens, method_position):
    """Count cost through a stop; use the full total as a bound for incomplete traces."""
    import warnings
    # Serialized totals define the accounting convention for this run.
    # Legacy runs can log trial probes without charging verifier tokens.
    total_tokens = float(total_tokens or 0)
    if total_tokens == 0:
        return 0.0

    traces = []
    if isinstance(logs, dict):
        for key in VERIFIER_ATTEMPT_LOG_KEYS:
            attempts = logs.get(key, [])
            if not isinstance(attempts, list):
                continue
            for attempt in attempts:
                if not isinstance(attempt, dict):
                    continue
                position = attempt.get("position")
                token_count = attempt.get("completion_tokens")
                if position is None or token_count is None:
                    continue
                traces.append((float(position), float(token_count)))

    total_tokens = float(total_tokens or 0)
    if not traces:
        if total_tokens == 0 or float(method_position) <= float(stop_position):
            return total_tokens
        warnings.warn(
            "Verifier trace missing: charging the full serialized verifier cost "
            "as a conservative upper bound at the hybrid stop.",
            RuntimeWarning,
        )
        return total_tokens

    trace_total = sum(tokens for _, tokens in traces)
    if not np.isclose(trace_total, total_tokens):
        if float(method_position) <= float(stop_position):
            # The whole monitor cost is known to have occurred by the stop.
            return total_tokens
        warnings.warn(
            "Verifier trace and serialized total differ: charging the full "
            "serialized verifier cost as a conservative upper bound at the hybrid stop.",
            RuntimeWarning,
        )
        return total_tokens
    return sum(tokens for position, tokens in traces if position <= float(stop_position))


def _agreement_vote_count(method_count, agreement_t):
    """Return the integer quorum implied by an agreement fraction."""
    if not 0 < float(agreement_t) <= 1:
        raise ValueError("agreement_t must be in (0, 1].")
    if method_count <= 0:
        raise ValueError("Every threshold stage needs at least one method.")
    return int(np.ceil(float(agreement_t) * method_count - 1e-12))


def simulate_threshold_exit_agreement(
    exit_df,
    threshold_mix,
    hard_stop_method=HARD_STOP_METHOD,
    position_col="thinking_tokens",
    require_stopped_early=True,
    agreement_cutoff=None,
    agreement_t=1.0,
):
    """Simulate the same cascade using exit votes, not answer equality.

    At each stage, at least ceil(agreement_t * N) required methods must
    have emitted a valid exit before the next stage begins. The latest
    available exiting method with an answer supplies the output; answer
    values are not compared. If no quorum fires, the baseline_16K answer is
    returned and counted as a normal early-exit result.
    """
    required_columns = {
        "benchmark", "model", "sample_id", "run_id",
        "method", "answer_norm", "correct", "stopped_early",
        "generation_failed", position_col, "completion_tokens",
        "answer_tokens",
        "verifier_completion_tokens", "verifier_logs",
    }
    missing = required_columns.difference(exit_df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    stages = [
        (int(threshold), tuple(methods))
        for threshold, methods in sorted(threshold_mix.items())
    ]
    if not stages:
        raise ValueError("threshold_mix cannot be empty.")
    if agreement_cutoff is not None and agreement_cutoff <= 0:
        raise ValueError("agreement_cutoff must be positive.")
    for _, methods in stages:
        _agreement_vote_count(len(methods), agreement_t)

    hard_stop_rows = resolve_hard_stop_rows(
        exit_df, hard_stop_method=hard_stop_method
    )
    unit_cols = ["benchmark", "model", "sample_id", "run_id"]
    hard_stop_lookup = {
        tuple(row[column] for column in unit_cols): row
        for _, row in hard_stop_rows.iterrows()
    }
    results = []

    for unit_key, run_rows in exit_df.groupby(
        unit_cols, dropna=False
    ):
        if unit_key not in hard_stop_lookup:
            continue

        hard_stop = hard_stop_lookup[unit_key]
        hard_stop_position = float(hard_stop[position_col])
        cascade_end = min(
            hard_stop_position,
            float(agreement_cutoff)
            if agreement_cutoff is not None
            else hard_stop_position,
        )
        by_method = {
            method: group.iloc[0]
            for method, group in run_rows.groupby("method")
        }

        selected_stage = None
        selected_methods = []
        selected_output = None
        decision_position = None
        methods_evaluated = set()
        selected_required_votes = None
        selected_vote_count = 0

        for stage_index, (stage_start, required_methods) in enumerate(stages):
            if stage_start >= cascade_end:
                break

            stage_end = (
                stages[stage_index + 1][0]
                if stage_index + 1 < len(stages)
                else cascade_end
            )
            stage_end = min(stage_end, cascade_end)
            methods_evaluated.update(required_methods)

            candidate_rows = pd.DataFrame([
                by_method[method] for method in required_methods
                if method in by_method
            ]).reset_index(drop=True)
            required_votes = _agreement_vote_count(
                len(required_methods), agreement_t
            )
            if candidate_rows.empty:
                continue
            valid_exit = (
                ~candidate_rows["generation_failed"].astype(bool)
                & candidate_rows[position_col].notna()
            )
            if require_stopped_early:
                valid_exit &= candidate_rows["stopped_early"].astype(bool)
            eligible_rows = candidate_rows[
                valid_exit
                & candidate_rows[position_col].lt(stage_end)
            ].sort_values([position_col, "method"])
            if len(eligible_rows) < required_votes:
                continue

            quorum_rows = eligible_rows.iloc[:required_votes].copy()
            current_decision_position = max(
                float(stage_start),
                float(quorum_rows[position_col].max()),
            )
            # Exit agreement does not inspect or compare answer values.
            output_candidates = eligible_rows[
                eligible_rows["answer_norm"].notna()
                & eligible_rows[position_col].le(current_decision_position)
            ]
            if output_candidates.empty:
                later_outputs = eligible_rows[
                    eligible_rows["answer_norm"].notna()
                ]
                if later_outputs.empty:
                    continue
                selected_output = later_outputs.iloc[0]
                current_decision_position = max(
                    current_decision_position,
                    float(selected_output[position_col]),
                )
            else:
                selected_output = output_candidates.iloc[-1]
            selected_stage = stage_start
            selected_methods = quorum_rows["method"].tolist()
            selected_required_votes = required_votes
            selected_vote_count = len(quorum_rows)
            decision_position = current_decision_position
            break

        if selected_stage is not None:
            active_monitors = run_rows[
                run_rows["method"].isin(methods_evaluated)
            ].drop_duplicates("method")
            verifier_cost = sum(
                verifier_completion_tokens_through(
                    monitor["verifier_logs"], decision_position,
                    monitor["verifier_completion_tokens"], monitor[position_col],
                )
                for _, monitor in active_monitors.iterrows()
            )
            # Exit votes require no trial-answer comparison. Only the
            # method that supplies the returned answer incurs answer cost.
            answer_completion_cost = float(selected_output["answer_tokens"])
            hybrid_thinking_tokens = float(decision_position)
            hybrid_total_tokens = (
                hybrid_thinking_tokens
                + answer_completion_cost
                + verifier_cost
            )
            chosen_answer = selected_output["answer_norm"]
            hybrid_correct = bool(selected_output["correct"])
            selected_output_method = selected_output["method"]
            used_fallback = False
        else:
            scheduled_methods = {
                method
                for stage_start, methods in stages
                if stage_start < cascade_end
                for method in methods
            }
            active_monitors = run_rows[
                run_rows["method"].isin(scheduled_methods)
            ].drop_duplicates("method")
            verifier_cost = sum(
                verifier_completion_tokens_through(
                    monitor["verifier_logs"], cascade_end,
                    monitor["verifier_completion_tokens"], monitor[position_col],
                )
                for _, monitor in active_monitors.iterrows()
            )
            hard_stop_method_name = hard_stop["selected_hard_stop_method"]
            if hard_stop_method_name not in active_monitors["method"].values:
                verifier_cost += float(
                    hard_stop["verifier_completion_tokens"]
                )
            answer_completion_cost = 0.0
            hybrid_thinking_tokens = hard_stop_position
            # The hard-stop method is an ordinary 16K early-exit method. Its
            # recorded completion already includes its returned answer.
            hybrid_total_tokens = (
                float(hard_stop["completion_tokens"])
                + verifier_cost
            )
            chosen_answer = hard_stop["answer_norm"]
            hybrid_correct = bool(hard_stop["correct"])
            selected_output_method = hard_stop_method_name
            used_fallback = False
            selected_stage = hard_stop_position

        result = {
            column: value for column, value in zip(unit_cols, unit_key)
        }
        result.update({
            "policy": "threshold_exit_agreement_hybrid",
            "correct": hybrid_correct,
            "answer_norm": chosen_answer,
            "used_fallback": used_fallback,
            "accepted_stage": selected_stage,
            "agreement_methods": tuple(selected_methods),
            "agreement_t": float(agreement_t),
            "required_votes": selected_required_votes,
            "votes_received": selected_vote_count,
            "selected_output_method": selected_output_method,
            "thinking_tokens": hybrid_thinking_tokens,
            "answer_tokens_est": answer_completion_cost,
            "completion_methods": (selected_output_method,),
            "verifier_tokens_est": verifier_cost,
            "total_cost_tokens_est": hybrid_total_tokens,
            "hard_stop_method": hard_stop["selected_hard_stop_method"],
            "hard_stop_correct": bool(hard_stop["correct"]),
            "hard_stop_thinking_tokens": hard_stop_position,
            "hard_stop_total_tokens_est": float(
                hard_stop["total_cost_tokens_est"]
            ),
            "correctness_label_conflict": False,
        })
        results.append(result)

    return pd.DataFrame(results), hard_stop_rows

## Load trajectories and run the policy grid

A policy is evaluated only for models containing every requested voter and the hard-stop method. Missing run-level voters still cannot form quorum on that run; the 16K hard stop applies. The detailed run table includes the selected output method and cost components.


In [5]:
records = glob_all_outputs(INPUT_DIR)
if not records:
    raise ValueError(f"No evaluation records found under {INPUT_DIR}")
exit_df = build_exit_dataframe(records)
if exit_df.empty:
    raise ValueError("No method results found in the evaluation records")
selected_models = MODEL_CHOICES or sorted(exit_df["model"].dropna().unique())
exit_df = exit_df.loc[exit_df["model"].isin(selected_models)].copy()
if exit_df.empty:
    raise ValueError(f"No method results for MODEL_CHOICES={selected_models}")

policy_runs = []
skipped = []
for model, (config_name, methods), agreement_t in product(selected_models, METHOD_SETS.items(), AGREEMENT_FRACTIONS):
    model_df = exit_df.loc[exit_df["model"] == model]
    available = set(model_df["method"].dropna())
    missing = set(methods) | {HARD_STOP_METHOD}
    missing -= available
    if missing:
        skipped.append({"model": model, "config": config_name, "agreement_t": agreement_t, "missing_methods": ", ".join(sorted(missing))})
        continue
    threshold_mix = {stage: methods for stage in STAGES}
    runs, _hard_stops = simulate_threshold_exit_agreement(
        model_df, threshold_mix, hard_stop_method=HARD_STOP_METHOD,
        position_col="thinking_tokens", require_stopped_early=True,
        agreement_cutoff=AGREEMENT_CUTOFF, agreement_t=agreement_t,
    )
    if runs.empty:
        skipped.append({"model": model, "config": config_name, "agreement_t": agreement_t, "missing_methods": "no runs with a hard stop"})
        continue
    runs = runs.assign(config=config_name, methods=",".join(methods), agreement_t=agreement_t)
    policy_runs.append(runs)

grid_runs = pd.concat(policy_runs, ignore_index=True) if policy_runs else pd.DataFrame()
skipped_configs = pd.DataFrame(skipped)
if grid_runs.empty:
    raise ValueError("No grid configurations produced runs; inspect skipped_configs and method names")
grid_runs.to_csv(OUTPUT_DIR / "hybrid_grid_runs.csv", index=False)
skipped_configs.to_csv(OUTPUT_DIR / "hybrid_grid_skipped.csv", index=False)
display(skipped_configs)
print(f"Evaluated {grid_runs[['model', 'config', 'agreement_t']].drop_duplicates().shape[0]} model/config/quorum combinations")


Found 57 JSONL files


Excluded code records containing dynasor, answer-coverage, or puma logs: 6
Unrecoverable JSON parse failures: 0
Token-limit failure records: 0
Unique question/model cases with a token-limit failure: 0
Loaded 58529 evaluation records.


/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conserv

/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conserv

/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(


...

/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(


/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conserv

/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(
/tmp/ipykernel_2636921/3725930307.py:69: RuntimeWarning: Verifier trace and serialized total differ: charging the full serialized verifier cost as a conservative upper bound at the hybrid stop.
  warnings.warn(


""


Evaluated 75 model/config/quorum combinations


## Question-balanced results

Accuracy and token cost are averaged across repeated runs of each question, then equally across questions within a benchmark. The model summary averages benchmarks equally. `hard_stop_rate` is the share of runs that reached the 16K hard stop.


In [6]:
grid_runs["hard_stop"] = grid_runs["selected_output_method"].eq(HARD_STOP_METHOD)
question_results = grid_runs.groupby(["model", "config", "methods", "agreement_t", "benchmark", "sample_id"], as_index=False).agg(
    accuracy=("correct", "mean"),
    mean_tokens=("total_cost_tokens_est", "mean"),
    hard_stop_rate=("hard_stop", "mean"),
    runs=("run_id", "size"),
)
benchmark_results = question_results.groupby(["model", "config", "methods", "agreement_t", "benchmark"], as_index=False).agg(
    questions=("sample_id", "nunique"),
    accuracy=("accuracy", "mean"),
    mean_tokens=("mean_tokens", "mean"),
    hard_stop_rate=("hard_stop_rate", "mean"),
)
model_results = benchmark_results.groupby(["model", "config", "methods", "agreement_t"], as_index=False).agg(
    benchmarks=("benchmark", "nunique"),
    accuracy=("accuracy", "mean"),
    mean_tokens=("mean_tokens", "mean"),
    hard_stop_rate=("hard_stop_rate", "mean"),
)
model_results = model_results.sort_values(["model", "accuracy", "mean_tokens"], ascending=[True, False, True]).reset_index(drop=True)
for name, frame in [("question", question_results), ("benchmark", benchmark_results), ("model", model_results)]:
    frame.to_csv(OUTPUT_DIR / f"hybrid_grid_{name}_results.csv", index=False)
display(model_results)


,model,config,methods,agreement_t,benchmarks,accuracy,mean_tokens,hard_stop_rate
0,gemma4e4,without_codestop,"dtsr,deer,entrocut",0.75,8,0.560539,2668.138799,0.986205
1,gemma4e4,without_codestop,"dtsr,deer,entrocut",1.00,8,0.560539,2668.138799,0.986205
2,gemma4e4,all_four,"codestop,dtsr,deer,entrocut",1.00,8,0.560539,2685.947428,0.986205
3,gemma4e4,with_terminator,"codestop,dtsr,deer,entrocut,terminator_hf",1.00,8,0.560539,2686.619979,0.986908
4,gemma4e4,deer_entrocut,"deer,entrocut",0.75,8,0.555955,2561.968804,0.891650
...,...,...,...,...,...,...,...,...
70,r1_qwen3_8b,without_codestop,"dtsr,deer,entrocut",0.50,8,0.609693,4895.323222,0.371535
71,r1_qwen3_8b,without_dtsr,"codestop,deer,entrocut",0.50,8,0.605796,4732.033423,0.349420
72,r1_qwen3_8b,deer_entrocut,"deer,entrocut",0.50,8,0.605185,4693.500474,0.348499
73,r1_qwen3_8b,with_terminator,"codestop,dtsr,deer,entrocut,terminator_hf",0.50,8,0.535767,2665.002612,0.239592


## Select the best policy by average TITE

The score uses the same gain-target TITE definition as `EDA_hung.ipynb`. For each repeated run, the oracle is the cheapest correct result among CoDEStop, DEER, EntroCut, DTSR, finite baselines, and replay_eval; unsolved runs use an incorrect replay_eval fallback. A run is scored only when both `baseline_no_thinking` and `replay_eval` are available. TITE is computed per benchmark and averaged equally across benchmarks. The best policy for each model has the highest average TITE among configurations with the largest available valid benchmark scope; mean tokens break ties.


In [7]:
ORACLE_METHOD = "oracle"

def _build_token_requirement_table(
    subset,
    *,
    token_col,
    accuracy_col,
    oracle_method=ORACLE_METHOD,
    targets,
):
    """Return per-method token requirements for explicit accuracy targets."""
    required = {"method", "sample_id", token_col, accuracy_col}
    missing = required.difference(subset.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    methods = sorted(subset["method"].dropna().unique())
    if oracle_method not in methods:
        raise ValueError(f"Missing oracle method {oracle_method!r}")

    n_questions = subset["sample_id"].nunique()
    curves = {}
    for method in methods:
        per_question = (
            subset[subset["method"].eq(method)]
            .groupby("sample_id", as_index=False)
            .agg(
                success_probability=(accuracy_col, "mean"),
                tokens=(token_col, "mean"),
            )
        )
        per_question["success_probability"] = (
            per_question["success_probability"].fillna(0).clip(0, 1)
        )
        successful = per_question[
            per_question["success_probability"].gt(0)
            & np.isfinite(per_question["tokens"])
            & per_question["tokens"].ge(0)
        ].sort_values("tokens")
        curves[method] = (
            successful["tokens"].to_numpy(float),
            np.cumsum(
                successful["success_probability"].to_numpy(float) / n_questions
            ),
        )

    oracle_curve = curves[oracle_method][1]
    oracle_accuracy = float(oracle_curve[-1]) if len(oracle_curve) else 0.0
    target_values = np.asarray(targets, dtype=float)
    common_targets = np.sort(np.unique(target_values[
        np.isfinite(target_values)
        & (target_values > 0)
        & (target_values <= oracle_accuracy + 1e-12)
    ]))
    requirements = pd.DataFrame(
        index=methods, columns=common_targets, dtype=float
    )
    for method, (token_values, cumulative_accuracy) in curves.items():
        indices = np.searchsorted(
            cumulative_accuracy, common_targets, side="left"
        )
        requirements.loc[method] = [
            token_values[index] if index < len(token_values) else np.inf
            for index in indices
        ]
    return requirements



ORACLE_METHOD = "oracle"
TITE_NO_THINKING_METHOD = "baseline_no_thinking"
TITE_EPSILON = 0.001
TITE_GAIN_TARGETS = np.arange(1, 1001, dtype=float) / 1000

def _method_accuracy_on_scope(subset, method, accuracy_col, n_questions):
    rows = subset[subset["method"].eq(method)]
    if rows.empty or n_questions <= 0:
        return np.nan
    per_question = (
        rows.groupby("sample_id")[accuracy_col]
        .mean().fillna(0).clip(0, 1)
    )
    return float(per_question.sum() / n_questions)


def calculate_tite(
    subset,
    token_col="total_cost_tokens_est",
    accuracy_col=None,
    oracle_method=ORACLE_METHOD,
    no_thinking_method=TITE_NO_THINKING_METHOD,
    gain_targets=TITE_GAIN_TARGETS,
    epsilon=TITE_EPSILON,
    return_details=False,
):
    """Calculate TITE; unreached positive reasoning-gain targets score zero."""
    accuracy_col = accuracy_col or (
        "run_accuracy" if "run_accuracy" in subset else "correct"
    )
    methods = sorted(subset["method"].dropna().unique())
    n_questions = subset["sample_id"].nunique()
    accuracies = {
        method: _method_accuracy_on_scope(
            subset, method, accuracy_col, n_questions
        )
        for method in methods
    }
    missing_anchors = [
        method for method in [no_thinking_method, oracle_method]
        if method not in methods
    ]
    if missing_anchors:
        summary = pd.DataFrame({
            "method": methods,
            "TITE": np.nan,
            "accuracy": [accuracies[method] for method in methods],
            "efficiency_when_reached": np.nan,
            "geometric_token_multiplier": np.nan,
            "target_coverage": np.nan,
            "targets_reached": 0,
            "n_targets": 0,
            "first_target": np.nan,
            "last_target": np.nan,
            "status": "Missing TITE anchor",
            "gain_coverage": np.nan,
            "excess_accuracy": np.nan,
            "no_thinking_accuracy": np.nan,
            "oracle_accuracy": np.nan,
            "reasoning_gain": np.nan,
            "rg_applicable": False,
        })
        detail = pd.DataFrame()
        return (summary, detail) if return_details else summary

    no_thinking_accuracy = accuracies[no_thinking_method]
    oracle_accuracy = accuracies[oracle_method]
    reasoning_gain = oracle_accuracy - no_thinking_accuracy
    applicable = bool(np.isfinite(reasoning_gain) and reasoning_gain > epsilon)

    summary_rows = []
    detail_rows = []
    if applicable:
        gain_targets = np.asarray(gain_targets, dtype=float)
        gain_targets = np.sort(np.unique(gain_targets[
            np.isfinite(gain_targets)
            & (gain_targets > 0)
            & (gain_targets <= 1)
        ]))
        accuracy_targets = no_thinking_accuracy + gain_targets * reasoning_gain
        requirements = _build_token_requirement_table(
            subset,
            token_col=token_col,
            accuracy_col=accuracy_col,
            oracle_method=oracle_method,
            targets=accuracy_targets,
        )
        oracle_tokens = requirements.loc[oracle_method].to_numpy(float)
        if len(requirements.columns) != len(gain_targets):
            raise ValueError(
                "The optimal oracle does not reach its own accuracy anchor"
            )

        for method in methods:
            method_tokens = requirements.loc[method].to_numpy(float)
            reached = np.isfinite(method_tokens) & np.isfinite(oracle_tokens)
            target_efficiency = np.zeros(len(gain_targets), dtype=float)
            target_efficiency[reached] = np.minimum(
                oracle_tokens[reached]
                / np.maximum(method_tokens[reached], 1e-12),
                1.0,
            )
            gain_coverage = float(np.clip(
                (accuracies[method] - no_thinking_accuracy) / reasoning_gain,
                0, 1,
            ))
            target_coverage = float(reached.mean()) if len(reached) else 0.0
            multiplier = (
                float(np.exp(np.mean(np.log(
                    np.maximum(method_tokens[reached], 1e-12)
                    / np.maximum(oracle_tokens[reached], 1e-12)
                ))))
                if reached.any()
                else np.nan
            )
            summary_rows.append({
                "method": method,
                "TITE": float(target_efficiency.mean()),
                "accuracy": accuracies[method],
                "efficiency_when_reached": (
                    float(target_efficiency[reached].mean())
                    if reached.any() else np.nan
                ),
                "geometric_token_multiplier": multiplier,
                "target_coverage": target_coverage,
                "targets_reached": int(reached.sum()),
                "n_targets": len(gain_targets),
                "first_target": accuracy_targets.min(),
                "last_target": accuracy_targets.max(),
                "status": (
                    "Full gain coverage" if target_coverage == 1
                    else "Partial gain coverage" if target_coverage > 0
                    else "No gain target reached"
                ),
                "gain_coverage": gain_coverage,
                "excess_accuracy": float(max(
                    0, accuracies[method] - oracle_accuracy
                )),
                "no_thinking_accuracy": no_thinking_accuracy,
                "oracle_accuracy": oracle_accuracy,
                "reasoning_gain": reasoning_gain,
                "rg_applicable": True,
            })
            detail_rows.extend({
                "method": method,
                "gain_target": gain_target,
                "accuracy_target": accuracy_target,
                "oracle_tokens": oracle_cost,
                "method_tokens": method_cost if is_reached else np.nan,
                "target_efficiency": efficiency,
                "reached": bool(is_reached),
            } for gain_target, accuracy_target, oracle_cost, method_cost,
                efficiency, is_reached in zip(
                    gain_targets, accuracy_targets, oracle_tokens, method_tokens,
                    target_efficiency, reached
                ))
    else:
        for method in methods:
            summary_rows.append({
                "method": method, "TITE": np.nan,
                "accuracy": accuracies[method],
                "efficiency_when_reached": np.nan,
                "geometric_token_multiplier": np.nan,
                "target_coverage": np.nan,
                "targets_reached": 0, "n_targets": 0,
                "first_target": np.nan, "last_target": np.nan,
                "status": "Reasoning gain not applicable",
                "gain_coverage": np.nan, "excess_accuracy": np.nan,
                "no_thinking_accuracy": no_thinking_accuracy,
                "oracle_accuracy": oracle_accuracy,
                "reasoning_gain": reasoning_gain, "rg_applicable": False,
            })

    summary = pd.DataFrame(summary_rows)
    detail = pd.DataFrame(detail_rows)
    return (summary, detail) if return_details else summary


In [8]:
# Build the same optimal-oracle candidate set used by EDA_hung.
ORACLE_CANDIDATES = {"codestop", "deer", "entrocut", "dtsr", "replay_eval"}
unit_columns = ["benchmark", "model", "sample_id", "run_id"]
reference_rows = []
for key, group in exit_df.groupby(unit_columns, dropna=False):
    no_thinking = group.loc[group["method"].eq(TITE_NO_THINKING_METHOD)]
    replay = group.loc[group["method"].eq("replay_eval")]
    if no_thinking.empty or replay.empty:
        continue
    candidates = group.loc[
        group["method"].isin(ORACLE_CANDIDATES)
        | group["method"].str.match(r"baseline_(?!no_thinking)", na=False)
    ]
    correct_candidates = candidates.loc[candidates["correct"].eq(True) & candidates["total_cost_tokens_est"].notna()]
    if correct_candidates.empty:
        oracle = replay.iloc[0].copy()
        oracle["correct"] = False
    else:
        oracle = correct_candidates.loc[correct_candidates["total_cost_tokens_est"].idxmin()].copy()
    oracle["method"] = ORACLE_METHOD
    reference_rows.extend([no_thinking.iloc[0].copy(), oracle])
reference_df = pd.DataFrame(reference_rows)
if reference_df.empty:
    raise ValueError("TITE needs baseline_no_thinking and replay_eval rows on the same runs")

score_rows = []
for (model, config, agreement_t, benchmark), policy in grid_runs.groupby(["model", "config", "agreement_t", "benchmark"], sort=False):
    anchors = reference_df.loc[(reference_df["model"] == model) & (reference_df["benchmark"] == benchmark)]
    anchor_runs = set(anchors.loc[anchors["method"].eq(ORACLE_METHOD), "run_id"])
    scored = policy.loc[policy["run_id"].isin(anchor_runs)].copy()
    if scored.empty:
        continue
    aligned_anchors = anchors.loc[anchors["run_id"].isin(scored["run_id"])]
    scored["method"] = "grid_policy"
    evaluation = pd.concat([
        aligned_anchors[["method", "sample_id", "correct", "total_cost_tokens_est"]],
        scored[["method", "sample_id", "correct", "total_cost_tokens_est"]],
    ], ignore_index=True)
    result = calculate_tite(evaluation).set_index("method").loc["grid_policy"]
    score_rows.append({
        "model": model, "config": config, "methods": policy["methods"].iloc[0],
        "agreement_t": agreement_t, "benchmark": benchmark,
        "questions": scored["sample_id"].nunique(), "runs": len(scored),
        "TITE": result["TITE"], "TITE_status": result["status"],
        "accuracy": result["accuracy"],
        "mean_tokens": scored.groupby("sample_id")["total_cost_tokens_est"].mean().mean(),
    })
grid_tite_benchmark = pd.DataFrame(score_rows)
if grid_tite_benchmark.empty:
    raise ValueError("No grid policies had the required TITE anchor rows")
grid_tite_model = grid_tite_benchmark.groupby(["model", "config", "methods", "agreement_t"], as_index=False).agg(
    benchmarks=("benchmark", "nunique"), valid_benchmarks=("TITE", "count"),
    avg_TITE=("TITE", "mean"), avg_accuracy=("accuracy", "mean"),
    avg_tokens=("mean_tokens", "mean"),
)
# Compare policies only when they have TITE for every scored benchmark of that model.
max_scored_benchmarks = grid_tite_model.groupby("model")["valid_benchmarks"].transform("max")
grid_tite_model["complete_TITE_scope"] = grid_tite_model["valid_benchmarks"].eq(max_scored_benchmarks) & grid_tite_model["avg_TITE"].notna()
grid_tite_model = grid_tite_model.sort_values(["model", "complete_TITE_scope", "avg_TITE", "avg_tokens"], ascending=[True, False, False, True]).reset_index(drop=True)
best_TITE_policy = grid_tite_model.loc[grid_tite_model["complete_TITE_scope"]].groupby("model", sort=False).head(1).reset_index(drop=True)
for name, frame in [("benchmark", grid_tite_benchmark), ("model", grid_tite_model), ("best", best_TITE_policy)]:
    frame.to_csv(OUTPUT_DIR / f"hybrid_grid_tite_{name}.csv", index=False)
display(best_TITE_policy)
display(grid_tite_model)


,model,config,methods,agreement_t,benchmarks,valid_benchmarks,avg_TITE,avg_accuracy,avg_tokens,complete_TITE_scope
0,gemma4e4,deer_entrocut,"deer,entrocut",0.75,8,8,0.383865,0.555955,2561.968804,True
1,gpt_oss_20b_high,deer_entrocut,"deer,entrocut",0.50,8,8,0.304616,0.697257,3158.550114,True
2,qwen35_4b,deer_entrocut,"deer,entrocut",0.50,8,8,0.195178,0.601057,5522.234383,True
3,qwen35_9b,deer_entrocut,"deer,entrocut",0.75,8,8,0.133444,0.668952,5211.609179,True
4,r1_qwen3_8b,deer_entrocut,"deer,entrocut",0.50,8,8,0.235440,0.604831,4697.270627,True


,model,config,methods,agreement_t,benchmarks,valid_benchmarks,avg_TITE,avg_accuracy,avg_tokens,complete_TITE_scope
0,gemma4e4,deer_entrocut,"deer,entrocut",0.75,8,8,0.383865,0.555955,2561.968804,True
1,gemma4e4,deer_entrocut,"deer,entrocut",1.00,8,8,0.383865,0.555955,2561.968804,True
2,gemma4e4,without_dtsr,"codestop,deer,entrocut",0.50,8,8,0.383333,0.555330,2489.132759,True
3,gemma4e4,with_terminator,"codestop,dtsr,deer,entrocut,terminator_hf",0.50,8,8,0.383002,0.555086,2462.118290,True
4,gemma4e4,without_codestop,"dtsr,deer,entrocut",0.50,8,8,0.382883,0.555534,2522.812800,True
...,...,...,...,...,...,...,...,...,...,...
70,r1_qwen3_8b,without_codestop,"dtsr,deer,entrocut",1.00,8,8,0.220757,0.623010,6575.170546,True
71,r1_qwen3_8b,all_four,"codestop,dtsr,deer,entrocut",1.00,8,8,0.219743,0.623010,6610.298851,True
72,r1_qwen3_8b,with_terminator,"codestop,dtsr,deer,entrocut,terminator_hf",1.00,8,8,0.209685,0.623260,6731.366766,True
73,r1_qwen3_8b,with_terminator,"codestop,dtsr,deer,entrocut,terminator_hf",0.50,8,8,0.209096,0.535376,2669.445982,True


## Results for each selected policy

Show each benchmark and the benchmark-macro average for each benchmark group, for the best-TITE configuration selected above. Group averages use only benchmarks with applicable TITE; the counts show the available scope.


In [9]:
DATASET_GROUPS = {
    "math+qa": ["aime26", "gpqa_diamond", "gsm8k", "math500", "mmlu_pro_subset"],
    "code": ["humaneval", "livecodebench"],
    "hallucination": ["truthfulqa", "halueval"],
}

selection_cols = ["model", "config", "methods", "agreement_t"]
best_tite_benchmark = grid_tite_benchmark.merge(
    best_TITE_policy[selection_cols], on=selection_cols, how="inner", validate="many_to_one"
).sort_values(["model", "benchmark"]).reset_index(drop=True)

benchmark_to_group = {
    benchmark: group
    for group, benchmarks in DATASET_GROUPS.items()
    for benchmark in benchmarks
}
best_tite_benchmark["benchmark_group"] = best_tite_benchmark["benchmark"].map(benchmark_to_group).fillna("other")
best_tite_group = best_tite_benchmark.groupby(
    selection_cols + ["benchmark_group"], as_index=False
).agg(
    benchmarks=("benchmark", "nunique"),
    valid_benchmarks=("TITE", "count"),
    questions=("questions", "sum"),
    runs=("runs", "sum"),
    avg_TITE=("TITE", "mean"),
    avg_accuracy=("accuracy", "mean"),
    avg_tokens=("mean_tokens", "mean"),
).sort_values(["model", "benchmark_group"]).reset_index(drop=True)

best_tite_benchmark.to_csv(OUTPUT_DIR / "hybrid_grid_tite_best_benchmark.csv", index=False)
best_tite_group.to_csv(OUTPUT_DIR / "hybrid_grid_tite_best_group.csv", index=False)
for model, selected in best_TITE_policy.groupby("model", sort=False):
    print(f"Best TITE policy: {model}")
    display(selected[selection_cols + ["avg_TITE", "avg_accuracy", "avg_tokens"]])
    print("Benchmarks")
    display(best_tite_benchmark.loc[best_tite_benchmark["model"].eq(model),
        ["benchmark_group", "benchmark", "questions", "runs", "TITE", "TITE_status", "accuracy", "mean_tokens"]])
    print("Benchmark groups")
    display(best_tite_group.loc[best_tite_group["model"].eq(model),
        ["benchmark_group", "benchmarks", "valid_benchmarks", "questions", "runs", "avg_TITE", "avg_accuracy", "avg_tokens"]])


Best TITE policy: gemma4e4


,model,config,methods,agreement_t,avg_TITE,avg_accuracy,avg_tokens
0,gemma4e4,deer_entrocut,"deer,entrocut",0.75,0.383865,0.555955,2561.968804


Benchmarks


,benchmark_group,benchmark,questions,runs,TITE,TITE_status,accuracy,mean_tokens
0,math+qa,aime26,30,60,0.528402,Partial gain coverage,0.383333,5642.550000
1,math+qa,gpqa_diamond,198,395,0.262431,Partial gain coverage,0.497475,2945.684343
2,math+qa,gsm8k,1316,1316,0.745400,Partial gain coverage,0.587386,763.110942
3,hallucination,halueval,4507,4511,0.000000,No gain target reached,0.810628,367.162636
4,code,livecodebench,1055,1055,0.300632,Partial gain coverage,0.415166,6213.083412
5,math+qa,math500,500,999,0.631431,Partial gain coverage,0.824000,2063.626000
6,math+qa,mmlu_pro_subset,2392,2392,0.117552,Partial gain coverage,0.618311,1568.988712
7,hallucination,truthfulqa,811,1562,0.485070,Partial gain coverage,0.311344,931.544390


Benchmark groups


...

,model,config,methods,agreement_t,avg_TITE,avg_accuracy,avg_tokens
4,r1_qwen3_8b,deer_entrocut,"deer,entrocut",0.5,0.23544,0.604831,4697.270627


Benchmarks


,benchmark_group,benchmark,questions,runs,TITE,TITE_status,accuracy,mean_tokens
32,math+qa,aime26,30,90,0.504532,Partial gain coverage,0.611111,12900.544444
33,math+qa,gpqa_diamond,198,396,0.124671,Partial gain coverage,0.515152,6964.484848
34,math+qa,gsm8k,1319,1319,0.389961,Partial gain coverage,0.922669,1139.275967
35,hallucination,halueval,4507,4507,0.000000,No gain target reached,0.647659,334.285112
36,code,livecodebench,1055,1055,0.021321,Partial gain coverage,0.266351,8454.666351
37,math+qa,math500,500,500,0.418433,Partial gain coverage,0.802000,2805.472000
38,math+qa,mmlu_pro_subset,2230,2230,0.267182,Partial gain coverage,0.677130,3481.476682
39,hallucination,truthfulqa,817,817,0.157424,Partial gain coverage,0.396573,1497.959608


Benchmark groups


,benchmark_group,benchmarks,valid_benchmarks,questions,runs,avg_TITE,avg_accuracy,avg_tokens
12,code,1,1,1055,1055,0.021321,0.266351,8454.666351
13,hallucination,2,2,5324,5324,0.078712,0.522116,916.122360
14,math+qa,5,5,4277,4535,0.340956,0.705612,5458.250788
